# Decentralised MPC Logging to W&B

In this notebook, we run the Decentralised MPC controller and log its results to the shared W&B server.

The district load is logged during the simulation loop. This allows us to monitor the MPC tracking progress while the controller is running.

The KPIs and building temperature plot are logged at the end, because they require the full simulation trajectory.

In [1]:
import os
os.environ["WANDB_SILENT"] = "true"

import wandb
import plotly.graph_objects as go

WANDB_ENTITY="CityLearn-TeamB"
WANDB_PROJECT="CityLearn"

## Load Dataset and Simulation Window

The TX dataset simulation window corresponds to June.  
All arrays are simulation-relative, meaning index 0 corresponds to the first simulated hour.

In [2]:
import sys
from pathlib import Path

# 1) Find repo root = folder that contains "citylearn/"
HERE = Path.cwd()
REPO_DIR = None
for p in [HERE] + list(HERE.parents):
    if (p / "citylearn").exists():
        REPO_DIR = p
        break

if REPO_DIR is None:
    raise FileNotFoundError(
        "Couldn't find the repo root (a folder containing 'citylearn'). "
        "Make sure you cloned the repo and opened the notebook from somewhere inside it."
    )

# 2) Force local CityLearn (repo version)
sys.path.insert(0, str(REPO_DIR))
import citylearn
print("✅ citylearn loaded from:", citylearn.__file__)
print("✅ repo root:", REPO_DIR)

# 3) Dataset paths  – TX for the MPC comparison
CLIMATE = "TX"  # TX – cooling-dominated, June simulation window
DATASET_DIR = REPO_DIR / "data" / "datasets" / f"annex96_ce1_{CLIMATE.lower()}_neighborhood"
SCHEMA_PATH = DATASET_DIR / "schema.json"

print("\nClimate:", CLIMATE)
print("Dataset directory:", DATASET_DIR, " | exists:", DATASET_DIR.exists())
print("Schema path:", SCHEMA_PATH, " | exists:", SCHEMA_PATH.exists())

# 4) Add MPC module directory to the path
MPC_DIR = REPO_DIR / "MPC" / "mpc_claude"
if str(MPC_DIR) not in sys.path:
    sys.path.insert(0, str(MPC_DIR))

✅ citylearn loaded from: c:\Users\yusuf\Documents\Python_Projects\CITYLEARN - Q4\annex96_common_exercise_1\citylearn\__init__.py
✅ repo root: c:\Users\yusuf\Documents\Python_Projects\CITYLEARN - Q4\annex96_common_exercise_1

Climate: TX
Dataset directory: c:\Users\yusuf\Documents\Python_Projects\CITYLEARN - Q4\annex96_common_exercise_1\data\datasets\annex96_ce1_tx_neighborhood  | exists: True
Schema path: c:\Users\yusuf\Documents\Python_Projects\CITYLEARN - Q4\annex96_common_exercise_1\data\datasets\annex96_ce1_tx_neighborhood\schema.json  | exists: True


In [3]:
import pandas as pd
import warnings, logging
warnings.filterwarnings("ignore")
logging.disable(logging.INFO)

# TX simulation does NOT start at timestep 0 (Simulate in June).
sim_start = 3624
sim_end = 4343

district_target_df = pd.read_csv(DATASET_DIR / "district_target.csv")
district_target = district_target_df["district_load_target"].values[sim_start : sim_end + 1]

print(f"Simulation window : steps {sim_start} – {sim_end}  ({len(district_target)} hours)")

Simulation window : steps 3624 – 4343  (720 hours)


## Run Decentralized MPC

Here, we start one W&B run for the MPC controller.  
At every MPC step, the district load is logged immediately to W&B.

In [4]:
import numpy as np
from tqdm.notebook import tqdm
from MPC.mpc_decentralized_1.utils import make_env
from MPC.mpc_decentralized_1.mpc_agent import MPCAgent

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name="MPC_Decentralised_1",
    reinit=True,
    settings=wandb.Settings(console="off")
)

wandb.define_metric("hour")
wandb.define_metric("district_load", step_metric="hour")

env = make_env(central_agent=False)
obs, _ = env.reset()

agent_mpc = MPCAgent(
    env,
    horizon        = 24,
    w_track        = 50.0,
    w_comfort      = 20.0,
    w_smooth       = 1.0,
    w_net_smooth   = 50.0,
    w_terminal_soc = 5.0,
)

T_total = sim_end - sim_start + 1

with tqdm(total=T_total, desc="Decentralised MPC", unit="step") as pbar:

    while not env.terminated:

        actions = agent_mpc.predict(obs)
        obs, _, terminated, truncated, _ = env.step(actions)

        # Get latest district load for live W&B logging
        current_load = env.net_electricity_consumption[-1]

        t = len(env.net_electricity_consumption) - 1

        wandb.log({
            "hour": t,
            "district_load": float(current_load),
        })

        pbar.update(1)

        if terminated or truncated:
            break

# Extract final arrays from environment after simulation
district_load_mpc = np.array(env.net_electricity_consumption)

mpc_building_temps = np.array([
    b.energy_simulation.indoor_dry_bulb_temperature[:len(district_load_mpc)]
    for b in env.buildings
]).T

print("district_load_mpc shape:", district_load_mpc.shape)
print("mpc_building_temps shape:", mpc_building_temps.shape)

Decentralised MPC:   0%|          | 0/720 [00:00<?, ?step/s]

district_load_mpc shape: (720,)
mpc_building_temps shape: (720, 25)


## Log KPIs

After the simulation is complete, the MPC controller is evaluated using the same KPIs as the tutorial notebook.

In [6]:
from SERVER.KPIs import compute_kpis

kpis_mpc = compute_kpis(district_target, district_load_mpc, mpc_building_temps)

wandb.log({
    "NMBE [%]": float(kpis_mpc["NMBE [%]"]),
    "CV-RMSE [%]": float(kpis_mpc["CV-RMSE [%]"]),
    "Temp Comfort violation [%]": float(kpis_mpc["Temp Comfort violation [%]"]),
})

print(kpis_mpc)

{'NMBE [%]': 10.979, 'CV-RMSE [%]': 79.643, 'Temp Comfort violation [%]': 31.42}


## Log Building Temperature Plot

The indoor temperatures of all 25 buildings are logged as an interactive Plotly figure.

The plot contains:
- all individual building temperature trajectories,
- the mean indoor temperature,
- the comfort band between 22–26 °C.

In [7]:
T = mpc_building_temps.shape[0]

fig = go.Figure()

# Comfort band 22–26 °C
fig.add_hrect(
    y0=22,
    y1=26,
    fillcolor="green",
    opacity=0.12,
    line_width=0,
    annotation_text="Comfort band [22–26 °C]",
    annotation_position="top right"
)

fig.add_hline(y=22, line_dash="dash", line_color="red", line_width=1)
fig.add_hline(y=26, line_dash="dash", line_color="red", line_width=1)

# Individual building temperatures
for b in range(mpc_building_temps.shape[1]):

    fig.add_trace(go.Scatter(
        x=list(range(T)),
        y=mpc_building_temps[:, b],
        mode="lines",
        line=dict(color="rgba(70,130,180,0.2)", width=1),
        showlegend=False
    ))

# Mean temperature
mean_temp = mpc_building_temps.mean(axis=1)

fig.add_trace(go.Scatter(
    x=list(range(T)),
    y=mean_temp,
    mode="lines",
    line=dict(color="steelblue", width=3),
    name="Mean temperature"
))

fig.update_layout(
    title="Decentralised MPC - Indoor Temperature",
    xaxis_title="Hour",
    yaxis_title="Temperature [°C]"
)

wandb.log({
    "temperature_plot": wandb.Plotly(fig)
})

END LOGGING

In [8]:
wandb.finish()